<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio 1, Modulo 2 Tema 3 - Consumidor de streaming</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>

Este cuaderno lee los ficheros JSON que genera `M2T3_productor.ipynb` desde `input/crypto_prices` y calcula métricas con Spark Structured Streaming.

- generación de eventos: `M2T3_productor.ipynb`
- lectura y métricas: este cuaderno

## 1. Preparación del entorno

Ejecuta esta celda para crear las carpetas necesarias. Si ya existen, y tienen contenido es mejor eliminarlas

In [ ]:
import os
import shutil

INPUT_DIR = os.path.abspath("input/crypto_prices")
OUTPUT_DIR = os.path.abspath("output/crypto_prices_parquet")
CHECKPOINT_DIR = os.path.abspath("checkpoint/crypto_prices")

for path in (INPUT_DIR, OUTPUT_DIR, CHECKPOINT_DIR):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

print("Directorio de trabajo:", os.getcwd())
print("Carpetas preparadas y limpiadas:")
print("-", INPUT_DIR)
print("-", OUTPUT_DIR)
print("-", CHECKPOINT_DIR)


## 2. Inicialización de SparkSession

En Spark Structured Streaming se recomienda trabajar con `SparkSession`, ya que es el punto de entrada moderno para usar DataFrames, SQL y streaming estructurado.

Si estás ejecutando el ejercicio en modo local, `local[*]` utiliza todos los núcleos disponibles de la máquina.

In [ ]:
!pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder     .appName("CoinGeckoFileStreaming")     .master("local[*]")     .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

## 3. Definición del esquema

Cuando Spark lee JSON en streaming, es buena práctica definir explícitamente el esquema.

Esto evita que Spark tenga que inferirlo continuamente y hace el proceso más estable.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("coin", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("price", DoubleType(), True)
])

schema

## 4. Lectura del flujo de ficheros JSON

Spark leerá como streaming todos los nuevos ficheros que aparezcan en la carpeta `input/crypto_prices`.

Importante: los ficheros los genera `M2T3_productor.ipynb`; este cuaderno solo los consume y calcula métricas.


In [ ]:
stream_df = spark.readStream     .schema(schema)     .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss'Z'")     .json(INPUT_DIR)

stream_df.printSchema()


## 5. Consulta streaming básica en consola

Esta primera consulta muestra los eventos recibidos por Spark en la consola.

Sirve para comprobar que Spark detecta correctamente los nuevos ficheros JSON.

In [ ]:
raw_query = stream_df.writeStream     .format("console")     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "raw_console"))     .outputMode("append")     .option("truncate", "false")     .trigger(availableNow=True)     .start()
raw_query.awaitTermination()


## 6. Agregaciones por ventana temporal

Ahora se calculan métricas por criptomoneda en ventanas de 1 minuto:

- precio medio,
- precio máximo,
- precio mínimo,
- número de eventos procesados.

La marca de agua (`watermark`) permite gestionar eventos que lleguen tarde.

In [ ]:
from pyspark.sql.functions import col, window, avg, max, min, count

metrics_df = stream_df     .withWatermark("event_time", "1 minute")     .groupBy(
        window(col("event_time"), "1 minute"),
        col("coin")
    )     .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        count("*").alias("num_events")
    )

metrics_df.printSchema()

## 7. Visualización de métricas en consola

La salida en modo `complete` muestra la tabla agregada completa para que no aparezca vacía.


In [ ]:
metrics_query = metrics_df.writeStream     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "metrics_console"))     .outputMode("complete")     .format("console")     .option("truncate", "false")     .trigger(availableNow=True)     .start()
metrics_query.awaitTermination()


## 8. Escritura de eventos en Parquet

Parquet es un formato columnar eficiente, muy usado en arquitecturas de datos.

En streaming, Spark necesita una carpeta de checkpoint para mantener el estado del proceso y poder recuperarse ante fallos.

In [ ]:
parquet_query = stream_df.writeStream     .format("parquet")     .option("path", OUTPUT_DIR)     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "parquet"))     .outputMode("append")     .trigger(availableNow=True)     .start()
parquet_query.awaitTermination()


In [ ]:
# Para detener la consulta:
parquet_query.stop()

## 9. Lectura posterior de los datos Parquet

Cuando la consulta de escritura haya procesado algunos eventos, puedes detenerla y leer los datos almacenados en Parquet como un DataFrame estático.

In [ ]:
# Ejecuta esta celda después de detener parquet_query
# parquet_query.stop()

historical_df = spark.read.parquet(OUTPUT_DIR)
historical_df.show(truncate=False)
historical_df.groupBy("coin").count().show()

## 10. Detener consultas activas

Es importante detener las consultas streaming cuando se termina el ejercicio para liberar recursos y la instancia de spark.

In [ ]:
for query in spark.streams.active:
    print("Deteniendo consulta:", query.name, query.id)
    query.stop()

print("Consultas activas:", spark.streams.active)

spark.stop()

## 11. Ejercicios

1. A&ntilde;adir mas criptomonedas a la lista `COINS`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 1</strong></summary>

En el notebook productor, amplia la lista `COINS` con los identificadores usados por la API de CoinGecko. Por ejemplo:

```python
COINS = ["bitcoin", "ethereum", "solana", "cardano", "ripple"]
```

Despues vuelve a ejecutar el productor y deja activo el consumidor. El streaming deberia detectar nuevos eventos para las monedas anadidas si el productor las escribe en la carpeta de entrada.

</details>

2. Cambiar la moneda de referencia de `eur` a `usd`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 2</strong></summary>

En el productor, cambia el parametro de moneda de referencia usado en la llamada a CoinGecko. Si existe una variable como `VS_CURRENCY`, dejala asi:

```python
VS_CURRENCY = "usd"
```

Si el JSON contiene una columna como `currency`, comprueba en el consumidor que el nuevo valor llega correctamente:

```python
stream_df.select("coin", "currency", "price", "event_time").writeStream.format("console")
```

</details>

3. Calcular ventanas de 5 minutos en lugar de 1 minuto.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 3</strong></summary>

Sustituye la ventana temporal de 1 minuto por una de 5 minutos en la agregacion:

```python
metrics_5m_df = (
    parsed_df
    .withWatermark("event_time", "10 minutes")
    .groupBy(F.window("event_time", "5 minutes"), "coin")
    .agg(
        F.avg("price").alias("avg_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.count("*").alias("events")
    )
)
```

Una ventana de 5 minutos genera menos filas agregadas y suaviza mas la evolucion del precio que una ventana de 1 minuto.

</details>

4. Crear una alerta cuando el precio de Bitcoin supere un umbral definido.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 4</strong></summary>

Define un umbral y filtra los eventos de Bitcoin que lo superen:

```python
BITCOIN_THRESHOLD = 70000

alerts_df = (
    parsed_df
    .filter(F.col("coin") == "bitcoin")
    .filter(F.col("price") > BITCOIN_THRESHOLD)
    .select("event_time", "coin", "price")
)
```

Para verlo por consola:

```python
alerts_query = (
    alerts_df
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)
```

</details>

5. Guardar las metricas agregadas en Parquet en lugar de guardar los eventos originales.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 5</strong></summary>

Escribe el DataFrame agregado en Parquet, usando un checkpoint propio:

```python
metrics_query = (
    metrics_5m_df
    .writeStream
    .format("parquet")
    .option("path", "output/crypto_metrics_parquet")
    .option("checkpointLocation", "checkpoint/crypto_metrics_parquet")
    .outputMode("append")
    .start()
)
```

Para usar `append` con agregaciones por ventana, conviene definir `withWatermark`. Si no hay watermark, Spark puede necesitar `complete` y un sink compatible.

</details>

6. A&ntilde;adir una columna `source` con valor `coingecko`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 6</strong></summary>

Anade una columna literal despues de parsear los eventos:

```python
parsed_df = parsed_df.withColumn("source", F.lit("coingecko"))
```

Esta columna permite identificar el origen de los datos si en el futuro se integran otras APIs o fuentes de streaming.

</details>

7. Comparar el numero de eventos esperados con el numero de eventos realmente procesados.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 7</strong></summary>

Primero calcula el numero esperado segun el productor. Por ejemplo, si se generan `N` iteraciones y hay `len(COINS)` monedas:

```python
expected_events = N * len(COINS)
```

Despues cuenta los eventos procesados por Spark leyendo la salida historica o usando una agregacion streaming:

```python
processed_events = spark.read.parquet("output/crypto_prices").count()
print("Esperados:", expected_events)
print("Procesados:", processed_events)
print("Diferencia:", expected_events - processed_events)
```

Si hay diferencias, revisa si el productor seguia activo, si Spark habia procesado todos los ficheros, si hubo errores de esquema o si se reinicio la consulta con otro checkpoint.

</details>